## [控制流](https://docs.langchain.com/oss/python/langgraph/graph-api-control-flow)

### 分支结构 - 动态

节点的后续执行路径在`运行时`才确定，可以根据当前状态、输入数据或中间结果，动态决定要触发哪些下游任务或跳转到哪个下游节点

#### 一、并行分支

In [ ]:
import operator
from typing import Annotated, TypedDict, Sequence

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

TYPES = ["五言绝句", "七言绝句"]

class State(TypedDict):
    topic: str
    poems: Annotated[list, operator.add]

class WorkerState(TypedDict):
    topic: str
    type: str

class InputState(TypedDict):
    topic: str

def worker(state: WorkerState) -> State:
    poem = model.invoke(f"请写一首关于{state["topic"]}的{state["type"]}，无需解析内容")
    return {"poems": [poem.content]}

def conditional_edge(state: InputState) -> Sequence[Send]:
    return [
        Send("worker", {"topic": state["topic"], "type": poem_type})
        for poem_type in TYPES
    ]


builder = StateGraph(state_schema=State, input_schema=InputState)
builder.add_node(worker)
builder.add_conditional_edges(START, conditional_edge)
builder.add_edge("worker", END)
graph = builder.compile()
result = graph.invoke({"topic": "猫"})
print(result)

#### 二、条件分支

可以通过 command(goto=...) 在节点内部实现条件分支

与 add_conditional_edges 相比，command 更适合`状态更新`和`流程跳转`需要在一个节点返回值中的场景

In [ ]:
from typing import TypedDict, Literal
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

class State(TypedDict):
    topic: str
    style: str
    poem: str

def a(state: State):
    poem = model.invoke(f"请写一首关于{state["topic"]}的五言绝句，{state["style"]}，无需解析内容")
    return {"poem": poem.content}

def b(state: State):
    poem = model.invoke(f"请写一首关于{state["topic"]}的七言绝句，{state["style"]}，无需解析内容")
    return {"poem": poem.content}

def router(state: State) -> Command[Literal["a", "b", "__end__"]]:
    if state["style"] == "宋":
        return Command(goto="a", update={"style": "宋朝风格"})
    elif state["style"] == "唐":
        return Command(goto="b", update={"style": "唐朝风格"})
    else:
        return Command(goto=END)


builder = StateGraph(State)
builder.add_node(router)
builder.add_node(a)
builder.add_node(b)
builder.add_edge(START, "router")
builder.add_edge("a", END)
builder.add_edge("b", END)
graph = builder.compile()
result = graph.invoke({"topic": "猫", "style": "唐"})
print(result)
result = graph.invoke({"topic": "猫", "style": "宋"})
print(result)
result = graph.invoke({"topic": "猫", "style": "汉 "} )
print(result)
